In [7]:
%pip install azure-eventhub requests python-dotenv

   ---------------------------------------- 0.0/317.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/317.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/317.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/317.1 kB ? eta -:--:--
   ----- ---------------------------------- 41.0/317.1 kB 1.9 MB/s eta 0:00:01
   ------- ------------------------------- 61.4/317.1 kB 812.7 kB/s eta 0:00:01
   -------------- ----------------------- 122.9/317.1 kB 901.1 kB/s eta 0:00:01
   ----------------- -------------------- 143.4/317.1 kB 944.1 kB/s eta 0:00:01
   ----------------- -------------------- 143.4/317.1 kB 944.1 kB/s eta 0:00:01
   ----------------- -------------------- 143.4/317.1 kB 944.1 kB/s eta 0:00:01
   ------------------------ ------------- 204.8/317.1 kB 655.1 kB/s eta 0:00:01
   ------------------------ ------------- 204.8/317.1 kB 655.1 kB/s eta 0:00:01
   -------------------------------------- 317.1/317.1 kB 754.0 kB/s eta 0:00:


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os, json, time, random
from datetime import datetime, timezone
import requests
from dotenv import load_dotenv
from azure.eventhub import EventHubProducerClient, EventData

load_dotenv()
CONN_STR = os.environ["FABRIC_CUSTOM_ENDPOINT_CONNECTION_STRING"]

FEED_URL       = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
DURATA_SEC     = 600
PAUSA          = 60
MAX_EVENTI_SEC = 10
MAX_RETRY      = 5
BATCH_SIZE     = 20
print("Connection string caricata:", bool(CONN_STR))

Connection string caricata: True


In [8]:
def iso_utc(ms):
    return datetime.fromtimestamp(ms/1000, tz=timezone.utc).isoformat() if ms else None

def get_terremoti():
    for t in range(MAX_RETRY):
        try:
            r = requests.get(FEED_URL, timeout=15)
            r.raise_for_status()
            return r.json()["features"]
        except Exception as e:
            attesa = 2**t + random.random()
            print(f"Errore feed ({e}), retry tra {attesa:.1f}s")
            time.sleep(attesa)
    return []

def area_di(lon):
    if -170 <= lon < -30: return "americas"
    if -30 <= lon < 60:   return "europe_africa"
    return "asia_pacific"

def to_dict(f):
    p = f["properties"]
    lon, lat, depth = f["geometry"]["coordinates"]
    return {"event_id": f["id"], "event_time": iso_utc(p["time"]),
            "updated_time": iso_utc(p["updated"]), "magnitude": p["mag"],
            "mag_type": p["magType"], "place": p["place"],
            "latitude": lat, "longitude": lon, "depth_km": depth,
            "area": area_di(lon), "event_type": p["type"], "status": p["status"],
            "network": p["net"], "significance": p["sig"], "tsunami": p["tsunami"],
            "ingest_time": datetime.now(timezone.utc).isoformat()}

In [20]:
eventi = get_terremoti()
print("Eventi nel feed:", len(eventi))
with open("sample_events.json", "w", encoding="utf-8") as f:
    json.dump([to_dict(x) for x in eventi[:5]], f, indent=2)

Eventi nel feed: 209


In [9]:
producer = EventHubProducerClient.from_connection_string(CONN_STR)

def invia_batch(eventi):
    for t in range(MAX_RETRY):
        try:
            batch = producer.create_batch()
            for ev in eventi:
                try:
                    batch.add(EventData(json.dumps(ev)))
                except ValueError:
                    producer.send_batch(batch)
                    batch = producer.create_batch()
                    batch.add(EventData(json.dumps(ev)))
            if len(batch) > 0:
                producer.send_batch(batch)
            return True
        except Exception as e:
            attesa = 2**t + random.random()
            print(f"Invio fallito ({e}), retry tra {attesa:.1f}s")
            time.sleep(attesa)
    return False

In [ ]:
visti, totale, fine, ciclo = set(), 0, time.time() + DURATA_SEC, 0
try:
    while time.time() < fine:
        ciclo += 1
        nuovi = []
        for f in get_terremoti():
            ev = to_dict(f)
            chiave = (ev["event_id"], ev["updated_time"])
            if chiave not in visti:
                visti.add(chiave)
                nuovi.append(ev)

        for i in range(0, len(nuovi), BATCH_SIZE):
            blocco = nuovi[i:i+BATCH_SIZE]
            if invia_batch(blocco):
                totale += len(blocco)
            time.sleep(len(blocco) / MAX_EVENTI_SEC)

        print(f"Ciclo {ciclo}: nuovi {len(nuovi)} | totale inviati {totale}")
        time.sleep(PAUSA)
except KeyboardInterrupt:
    print("Interrotto")
finally:
    producer.close()

Ciclo 1: nuovi 205 | totale inviati 205
Ciclo 2: nuovi 1 | totale inviati 206
Ciclo 3: nuovi 0 | totale inviati 206


In [6]:
# Caricamento iniziale: eventi dell'ultima settimana
FEED_URL = (
    "https://earthquake.usgs.gov/earthquakes/"
    "feed/v1.0/summary/all_week.geojson"
)

eventi_settimana = [
    to_dict(f)
    for f in get_terremoti()
]

# Allineamento al filtro magnitude non nulla dell'Eventstream
eventi_settimana = [
    ev for ev in eventi_settimana
    if ev["magnitude"] is not None
]

print("Eventi disponibili:", len(eventi_settimana))

Eventi disponibili: 1845


In [ ]:
# Ricrea il producer
producer = EventHubProducerClient.from_connection_string(CONN_STR)

inviati = 0

try:
    for i in range(0, len(eventi_settimana), BATCH_SIZE):
        blocco = eventi_settimana[i:i + BATCH_SIZE]

        if not invia_batch(blocco):
            raise RuntimeError(
                "Invio non riuscito: caricamento interrotto."
            )

        inviati += len(blocco)
        print(
            f"Inviati {inviati}/{len(eventi_settimana)} eventi",
            flush=True
        )

        time.sleep(len(blocco) / MAX_EVENTI_SEC)

except KeyboardInterrupt:
    print("Caricamento interrotto dall'utente.")

finally:
    producer.close()
    print("Producer chiuso.")

Inviati 20/1845 eventi
Inviati 40/1845 eventi
Inviati 60/1845 eventi
Inviati 80/1845 eventi
Inviati 100/1845 eventi
Inviati 120/1845 eventi
Inviati 140/1845 eventi
Inviati 160/1845 eventi
Inviati 180/1845 eventi
Inviati 200/1845 eventi
Inviati 220/1845 eventi
Inviati 240/1845 eventi
Inviati 260/1845 eventi
Inviati 280/1845 eventi
Inviati 300/1845 eventi
Inviati 320/1845 eventi
Inviati 340/1845 eventi
Inviati 360/1845 eventi
Inviati 380/1845 eventi
Inviati 400/1845 eventi
Inviati 420/1845 eventi
Inviati 440/1845 eventi
Inviati 460/1845 eventi
Inviati 480/1845 eventi
Inviati 500/1845 eventi
Inviati 520/1845 eventi
Inviati 540/1845 eventi
Inviati 560/1845 eventi
Inviati 580/1845 eventi
Inviati 600/1845 eventi
Inviati 620/1845 eventi
Inviati 640/1845 eventi
Inviati 660/1845 eventi
Inviati 680/1845 eventi
Inviati 700/1845 eventi
Inviati 720/1845 eventi
Inviati 740/1845 eventi
Inviati 760/1845 eventi
Inviati 780/1845 eventi
Inviati 800/1845 eventi
Inviati 820/1845 eventi
Inviati 840/1845 eve

In [5]:
# Test completo con dati USGS reali dell'ultimo giorno
FEED_URL = (
    "https://earthquake.usgs.gov/earthquakes/"
    "feed/v1.0/summary/all_day.geojson"
)

eventi_test = [
    to_dict(f)
    for f in get_terremoti()
]

eventi_test = [
    ev for ev in eventi_test
    if ev["magnitude"] is not None
]

# Evita doppioni della stessa versione nella lista da inviare
eventi_test = list({
    (ev["event_id"], ev["updated_time"]): ev
    for ev in eventi_test
}.values())

sopra_soglia = [
    ev for ev in eventi_test
    if ev["magnitude"] > 5
]

print("Eventi da inviare:", len(eventi_test))
print("Eventi con magnitudo > 5:", len(sopra_soglia))

for ev in sopra_soglia:
    print(
        ev["event_id"],
        "| M:", ev["magnitude"],
        "| Data UTC:", ev["event_time"],
        "|", ev["place"]
    )

Eventi da inviare: 199
Eventi con magnitudo > 5: 12
us7000tty7 | M: 5.1 | Data UTC: 2026-10-29T08:49:42.448000+00:00 | 218 km NNW of Bamboo Flat, India
us7000ttxz | M: 6.4 | Data UTC: 2026-10-29T07:49:50.928000+00:00 | southeast of Easter Island
us7000ttxn | M: 5.4 | Data UTC: 2026-10-29T07:31:30.432000+00:00 | 114 km ENE of Barcelona, Philippines
us7000ttx8 | M: 5.1 | Data UTC: 2026-10-29T05:33:36.298000+00:00 | Balleny Islands region
us7000ttx0 | M: 5.4 | Data UTC: 2026-10-29T05:06:48.289000+00:00 | South Sandwich Islands region
us7000ttw5 | M: 5.7 | Data UTC: 2026-10-29T03:26:42.275000+00:00 | 82 km NE of Kiunga, Papua New Guinea
us7000ttw3 | M: 5.8 | Data UTC: 2026-10-29T03:19:05.288000+00:00 | 110 km ENE of Barcelona, Philippines
us7000ttw2 | M: 5.4 | Data UTC: 2026-10-29T03:17:54.507000+00:00 | 113 km ENE of Barcelona, Philippines
us7000ttvf | M: 5.4 | Data UTC: 2026-10-29T00:08:20.933000+00:00 | 109 km ENE of Luganville, Vanuatu
us7000ttua | M: 5.7 | Data UTC: 2026-10-28T22:06:4

In [18]:
producer = EventHubProducerClient.from_connection_string(CONN_STR)

inviati = 0
inizio_test = datetime.now(timezone.utc)

print("Inizio test UTC:", inizio_test.isoformat())

try:
    for i in range(0, len(eventi_test), BATCH_SIZE):
        blocco = eventi_test[i:i + BATCH_SIZE]

        if not invia_batch(blocco):
            raise RuntimeError(
                "Invio fallito: test interrotto."
            )

        inviati += len(blocco)

        print(
            f"Inviati {inviati}/{len(eventi_test)} eventi",
            flush=True
        )

        time.sleep(len(blocco) / MAX_EVENTI_SEC)

except KeyboardInterrupt:
    print("Test interrotto dall'utente.")

finally:
    producer.close()
    print("Producer chiuso.")
    print("Totale inviati:", inviati)

Inizio test UTC: 2026-10-30T17:20:55.463210+00:00
Inviati 20/199 eventi
Inviati 40/199 eventi
Inviati 60/199 eventi
Inviati 80/199 eventi
Inviati 100/199 eventi
Inviati 120/199 eventi
Inviati 140/199 eventi
Inviati 160/199 eventi
Inviati 180/199 eventi
Inviati 199/199 eventi
Producer chiuso.
Totale inviati: 199


In [19]:
from importlib.metadata import version
from pathlib import Path

pacchetti = ["azure-eventhub", "requests", "python-dotenv"]

contenuto = "\n".join(
    f"{nome}=={version(nome)}"
    for nome in pacchetti
) + "\n"

Path("requirements.txt").write_text(contenuto, encoding="utf-8")

print(contenuto)

azure-eventhub==5.15.1
requests==2.34.2
python-dotenv==1.2.4

